In [ ]:
# Cell 0 — Install correct versions (run first, then restart runtime)
!pip install -q "tensorflow==2.20.0" "keras==3.13.2"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/thesis_project'  # adjust to your Drive path

In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 17a: RF — Hyperparameter Search on MobileNetV2 Features
# Uses pre-extracted MobileNetV2 features from DDIM FashionMNIST
# Search on synth train/val features (no test set)
# Hyperparameter grid same as notebook 12a

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import itertools
import time

SEED = 42
np.random.seed(SEED)

# HP grid — same as notebook 12a
PARAM_GRID = {
    'criterion':    ['gini', 'entropy'],
    'max_depth':    [10, 50, 100],
    'n_estimators': [10, 50, 100],
}

FEATURES_DIR = os.path.join(PROJECT_DIR, 'data', 'extracted_features', 'seed_42', 'ddim')
RESULTS_DIR  = os.path.join(PROJECT_DIR, 'results', 'rf_features', 'hp_search')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Load Pre-extracted Features ──────────────────────────────────────

F_train = np.load(os.path.join(FEATURES_DIR, 'fashion_mnist_x_train_features.npy'))
y_train = np.load(os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim',
                               'fashion_mnist', 'fashion_mnist_synthetic.npz'))['y_train']

F_val   = np.load(os.path.join(FEATURES_DIR, 'fashion_mnist_x_val_features.npy'))
y_val   = np.load(os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim',
                               'fashion_mnist', 'fashion_mnist_synthetic.npz'))['y_val']

print(f'F_train: {F_train.shape}, F_val: {F_val.shape}')

In [ ]:
# ── Cell 3: Hyperparameter Search ────────────────────────────────────────────

keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*[PARAM_GRID[k] for k in keys]))
print(f'Total runs: {len(combos)}\n')

records = []

for i, values in enumerate(combos):
    params   = dict(zip(keys, values))
    run_name = f"crit{params['criterion']}_depth{params['max_depth']}_nest{params['n_estimators']}"

    clf = RandomForestClassifier(
        criterion=params['criterion'],
        max_depth=params['max_depth'],
        n_estimators=params['n_estimators'],
        random_state=SEED,
        n_jobs=-1
    )

    t0 = time.time()
    clf.fit(F_train, y_train)
    elapsed = time.time() - t0

    train_acc = accuracy_score(y_train, clf.predict(F_train))
    val_acc   = accuracy_score(y_val,   clf.predict(F_val))

    records.append({
        'run_name':     run_name,
        'criterion':    params['criterion'],
        'max_depth':    params['max_depth'],
        'n_estimators': params['n_estimators'],
        'train_acc':    round(train_acc, 4),
        'val_acc':      round(val_acc, 4),
        'fit_time_s':   round(elapsed, 1),
    })

    print(f'[{i+1:02d}/{len(combos)}] {run_name:45s}  '
          f'train={train_acc:.4f}  val={val_acc:.4f}  ({elapsed:.1f}s)')

results_df = pd.DataFrame(records)

In [ ]:
# ── Cell 4: Results ───────────────────────────────────────────────────────────

results_ranked = results_df.sort_values('val_acc', ascending=False).reset_index(drop=True)
print('Search results ranked by val_acc:')
print(results_ranked.to_string(index=False))

best = results_ranked.iloc[0]
print(f'\nBest run: {best["run_name"]}')
print(f'  criterion:    {best["criterion"]}')
print(f'  max_depth:    {best["max_depth"]}')
print(f'  n_estimators: {best["n_estimators"]}')
print(f'  val_acc:      {best["val_acc"]}')

results_df.to_csv(os.path.join(RESULTS_DIR, 'search_results.csv'), index=False)
results_ranked.to_csv(os.path.join(RESULTS_DIR, 'search_results_ranked.csv'), index=False)

In [ ]:
# ── Cell 5: Val Accuracy Plot ─────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 6))
run_names = results_ranked['run_name']
val_accs  = results_ranked['val_acc']
bars = ax.bar(range(len(run_names)), val_accs, color='steelblue', alpha=0.8)
bars[0].set_color('darkorange')
ax.set_xticks(range(len(run_names)))
ax.set_xticklabels(run_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Val Accuracy')
ax.set_title('Val Accuracy — All 18 RF Search Runs (DDIM FashionMNIST MobileNetV2 Features)')
ax.set_ylim(0, 1)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'val_accuracy_all_runs.png'), dpi=150, bbox_inches='tight')
plt.close()
print(f'Results saved to: {RESULTS_DIR}')